# Gamma basics

The Gamma API is Polymarket's catalogue: which events exist, which markets sit
inside them, and how they resolved. `pyolymarket` wraps it in two objects,
`Event` and `Market`.

This notebook walks through the 2024 US presidential election event. That
event is closed and resolved, so every number below is fixed and the notebook
reads the same whenever you run it.

In [1]:
import sys
from pathlib import Path

# Run straight from a clone, without installing the package first.
src = Path.cwd().parent / "src"
if src.is_dir():
    sys.path.insert(0, str(src))

import pyolymarket as pyoly

## Loading an event

An event is addressed either by its slug, which is the last path segment of
its polymarket.com URL, or by its numeric id. Either one costs a single
request.

In [2]:
event = pyoly.Event(slug = "presidential-election-winner-2024")
event

Event(slug='presidential-election-winner-2024', id='903193')

In [3]:
print(event.id)
print(event.title)
print(event.end_date)

903193
Presidential Election Winner 2024
2024-11-05 12:00:00+00:00


`end_date` is a timezone-aware `datetime`, not the raw string Gamma sends, so
it compares safely against `datetime.now(timezone.utc)`.

In [4]:
print(repr(event.data["endDate"]))
print(repr(event.end_date))

'2024-11-05T12:00:00Z'
datetime.datetime(2024, 11, 5, 12, 0, tzinfo=datetime.timezone.utc)


The same event by id, and the full payload underneath it. Anything the wrapper
does not expose as a property is still there in `.data`.

In [5]:
pyoly.Event(id = "903193")

Event(slug='presidential-election-winner-2024', id='903193')

In [6]:
print(len(event.data), "fields, including:")
sorted(event.data)[:12]

51 fields, including:


['$schema',
 'active',
 'archived',
 'automaticallyActive',
 'automaticallyResolved',
 'closed',
 'closedTime',
 'commentCount',
 'commentsEnabled',
 'createdAt',
 'creationDate',
 'cyom']

## The markets inside an event

`.markets` is free. The event payload already carries its markets in full, so
these `Market` objects are built from data that is already in hand rather than
fetched one by one.

In [7]:
markets = event.markets
print(len(markets), "markets")

for market in markets[:5]:
    print(market.slug)

17 markets
will-donald-trump-win-the-2024-us-presidential-election
will-joe-biden-win-the-2024-us-presidential-election
will-nikki-haley-win-the-2024-us-presidential-election
will-gavin-newsom-win-the-2024-us-presidential-election
will-robert-f-kennedy-jr-win-the-2024-us-presidential-election


`to_frame()` is the same list as a pandas DataFrame, one row per market.

In [8]:
event.to_frame()[["question", "conditionId", "outcomePrices"]].head()

,question,conditionId,outcomePrices
0,Will Donald Trump win the 2024 US Presidential...,0xdd22472e552920b8438158ea7238bfadfa4f736aa4ce...,"[""1"", ""0""]"
1,Will Joe Biden win the 2024 US Presidential El...,0x14018049e265a2d88f284be9588e2e3542e3a3df08cc...,"[""0"", ""1""]"
2,Will Nikki Haley win the 2024 US Presidential ...,0xced9f9d90c94db9f1e1dbd7d9fba82fe4fa7431c0d4e...,"[""0"", ""1""]"
3,Will Gavin Newsom win the 2024 US Presidential...,0x40bbdd26dc08406eedcb913efee7f7faddf50e16fc21...,"[""0"", ""1""]"
4,Will Robert F. Kennedy Jr. win the 2024 US Pre...,0x7da35195ac3c7bf167f88ab0c27067a99020e36de67d...,"[""0"", ""1""]"


## A single market

`Market` takes a slug or an id just like `Event` does.

In [9]:
trump = pyoly.Market(slug = "will-donald-trump-win-the-2024-us-presidential-election")

print(trump.outcomes)
print(trump.outcome_prices)
print(trump.condition_id)

['Yes', 'No']
[1.0, 0.0]
0xdd22472e552920b8438158ea7238bfadfa4f736aa4cee91a6b86c39ead110917


Gamma returns `outcomes`, `outcomePrices` and `clobTokenIds` as JSON-encoded
strings rather than as arrays. The properties above run the second decode pass
for you:

In [10]:
print(repr(trump.data["outcomePrices"]))
print(trump.outcome_prices)

'["1", "0"]'
[1.0, 0.0]


## Resolution

`resolved` compares the end date against now, and `resolution` reports what
the UMA oracle said. An open market answers `"pending"`; one that is past its
end date but not yet reported answers `"unresolved"`.

In [11]:
print(trump.resolved)
print(trump.resolution)

True
resolved


## Token ids, the bridge to the order book

Gamma identifies a market by slug or id, the CLOB by condition id and by one
token id per outcome. `token_id()` is the join between the two, and it accepts
either an outcome label or an index.

In [12]:
print(trump.token_ids)
print()
print(trump.token_id("Yes"))
print(trump.token_id(0))

['21742633143463906290569050155826241533067272736897614950488156847949938836455', '48331043336612883890938759509493159234755048973500640148014422747788308965732']

21742633143463906290569050155826241533067272736897614950488156847949938836455
21742633143463906290569050155826241533067272736897614950488156847949938836455


Labels are matched against `outcomes` rather than assuming index 0 is "Yes",
so an unrecognized label tells you what the market actually offers.

In [13]:
try:
    trump.token_id("Maybe")
except ValueError as error:
    print(error)

Unrecognized outcome 'Maybe'. Market outcomes are: Yes, No


## Refreshing

Both objects hold a snapshot. `refresh()` re-fetches it and stamps
`data_timestamp`, which is how you tell how stale a price is.

In [14]:
print(event.data_timestamp)
event.refresh()
print(event.data_timestamp)

2026-08-25 22:48:24.883242
2026-08-25 22:48:25.304906


## Errors

Failures raise typed exceptions rather than returning `None`, and they carry
the URL that produced them. `PolymarketNotFoundError` and
`PolymarketRateLimitError` both subclass `PolymarketAPIError`, so catching the
base class catches everything the API can throw at you.

In [15]:
try:
    pyoly.Event(slug = "an-event-that-does-not-exist")
except pyoly.PolymarketAPIError as error:
    print(type(error).__name__)
    print(error)

PolymarketNotFoundError
https://gamma-api.polymarket.com/events/slug/an-event-that-does-not-exist returned 404: {"type":"not found error","error":"slug not found"}
